In [1]:
# import packages
import os

import pandas as pd

from calendar import monthrange
from datetime import datetime, timedelta

from azureml.core import Dataset, Datastore, Workspace
from azureml.opendatasets import NoaaIsdWeather

Failure while loading azureml_run_type_providers. Failed to load entrypoint hyperdrive = azureml.train.hyperdrive:HyperDriveRun._from_run_dto with exception (azureml-telemetry 1.0.65 (/home/cody/miniconda3/envs/aml/lib/python3.7/site-packages), Requirement.parse('azureml-telemetry==0.1.0.6035949.*')).
Failure while loading azureml_run_type_providers. Failed to load entrypoint azureml.PipelineRun = azureml.pipeline.core.run:PipelineRun._from_dto with exception (azureml-core 1.0.65.1 (/home/cody/miniconda3/envs/aml/lib/python3.7/site-packages), Requirement.parse('azureml-core==0.1.0.6035949.*')).
Failure while loading azureml_run_type_providers. Failed to load entrypoint azureml.ReusedStepRun = azureml.pipeline.core.run:StepRun._from_reused_dto with exception (azureml-core 1.0.65.1 (/home/cody/miniconda3/envs/aml/lib/python3.7/site-packages), Requirement.parse('azureml-core==0.1.0.6035949.*')).
Failure while loading azureml_run_type_providers. Failed to load entrypoint azureml.StepRun = 

In [2]:
ws = Workspace.from_config()
dstore = ws.get_default_datastore()

In [ ]:
ws

In [ ]:
%%time 

target_years = list(range(2017, 2020))
start_month  = 1

for year in target_years:
    for month in range(start_month, 12+1):
        path = 'weather-data/{}/{:02d}/'.format(year, month)
        
        try:  
            start = datetime(year, month, 1)
            end   = datetime(year, month, monthrange(year, month)[1]) + timedelta(days=1)
            isd   = NoaaIsdWeather(start, end).to_pandas_dataframe()
            isd   = isd[isd['stationName'].str.contains('FLORIDA', regex=True, na=False)]
            
            os.makedirs(path, exist_ok=True)
            isd.to_parquet(path + 'data.parquet')
        except Exception as e:
            print('Month {} in year {} likely has no data.\n'.format(month, year))
            print('Exception: {}'.format(e))

ActivityStarted, to_pandas_dataframe
ActivityStarted, to_pandas_dataframe_in_worker
Target paths: ['/year=2017/month=1/', '/year=2017/month=2/']
Looking for parquet files...
Reading them into Pandas dataframe...
Reading ISDWeather/year=2017/month=1/part-00001-tid-1321158002197267978-8e3eb092-4b7a-42de-97ee-e23297ed8955-119.c000.snappy.parquet under container isdweatherdatacontainer
Reading ISDWeather/year=2017/month=2/part-00011-tid-1321158002197267978-8e3eb092-4b7a-42de-97ee-e23297ed8955-129.c000.snappy.parquet under container isdweatherdatacontainer
Done.
ActivityCompleted: Activity=to_pandas_dataframe_in_worker, HowEnded=Success, Duration=392256.3 [ms]
ActivityCompleted: Activity=to_pandas_dataframe, HowEnded=Success, Duration=392395.29 [ms]
ActivityStarted, to_pandas_dataframe
ActivityStarted, to_pandas_dataframe_in_worker
Target paths: ['/year=2017/month=2/', '/year=2017/month=3/']
Looking for parquet files...
Reading them into Pandas dataframe...
Reading ISDWeather/year=2017/mont

In [ ]:
# uploads everything in the folder 'data' to the datastore under a container named 'weather-data'
dstore.upload('weather-data', 'weather-data')